# Arabic Fatwa Preprocessing Pipeline (Multi-file → BM25 → RAG)

================================================================================
Arabic Fatwa Preprocessing Pipeline — Multi-file (title/question/answer)
                                        → BM25 → RAG
================================================================================
Author  : Senior Arabic NLP / IR / RAG Systems Engineer persona
Purpose : Discover, load, merge, clean, and normalize 21+ raw Excel (.xlsx)
          fatwa files (schema: title / question / answer only — no
          Category / Source in this dataset) into a single production-ready
          dataset for BM25 retrieval and RAG grounding.

Design philosophy (unchanged from the original single-file version):
    This pipeline is optimized for RETRIEVAL QUALITY, not classification.
      - We normalize aggressively (spelling variants must collapse together)
      - We do NOT stem/lemmatize/root-extract (root stemmers routinely
        collapse semantically DIFFERENT fiqh terms into the same root)
      - We keep two parallel tokenized versions (with/without stopwords)
        because BM25 behaves differently depending on stopword removal
      - We NEVER remove negation particles (لا / لم / لن / ما) or the
        prepositions (على / في / عن) from the "stopword-removed" version,
        because these single tokens can flip or narrow a fatwa's ruling
      - We never destructively alter Qur'an/Hadith markers embedded in
        fatwa answers (﴿ ﴾ ﷺ ؑ) — these are preserved through every step

WHAT CHANGED FOR THIS VERSION (multi-file title/question/answer schema):
    - STEP 1 now discovers and loads *21 .xlsx files* from a folder (instead
      of a single CSV) and merges them into one dataframe.
    - Column names are heterogeneous across files (Question / QUESTION /
      column1.question / fatwa_question / ...), so a configurable
      COLUMN_MAPPING dictionary standardizes every source file onto
      title / question / answer immediately after each file is loaded.
    - There is no Category/Source field in this dataset, so those columns
      and every reference to them have been removed from cleaning/export.
    - Missing titles are no longer grounds for row removal — a title is
      auto-generated from the question instead (generated_title flag).
    - Deduplication is now a three-stage process (title, then question,
      then answer) with per-stage counts printed for auditability.
    - Arabic normalization gained ؤ→و / ئ→ي (hamza-on-waw / hamza-on-yeh)
      collapsing, on top of the existing أ/إ/آ→ا and ى→ي rules.
    - Sentence-level punctuation (. , ، ؛ : ؟) is now intentionally KEPT
      (not stripped) to support future sentence segmentation / chunking /
      citation extraction — only noisy non-Arabic symbols are removed.
    - New retrieval_title / retrieval_question / retrieval_answer fields
      feed a structured retrieval_document with explicit [Title] /
      [Question] / [Answer] sections (titles often summarize the fatwa
      topic and materially improve BM25/RAG retrieval quality).
    - New needs_chunking flag (answer > 300 words) for the RAG chunking
      stage downstream.
    - Export now writes both clean_fatwas.xlsx AND clean_fatwas.csv
      (utf-8-sig), with a fixed, minimal column set.

Dependencies:
    pip install pandas openpyxl

Run:
    python fatwa_preprocessing_pipeline.py --input-folder raw_data --output-dir .
================================================================================

## Setup: Imports & Logging

In [49]:
from __future__ import annotations

import re
import argparse
import logging
import unicodedata
from pathlib import Path
from typing import Dict, List

import pandas as pd

# ------------------------------------------------------------------------------
# Logging setup (production-friendly: timestamps + level, single console handler)
# All STEP-by-STEP diagnostics required by the spec are emitted via logger.info()
# so they appear in the console exactly like the original print() statements,
# but are also compatible with log aggregation in a production deployment.
# ------------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("fatwa_pipeline")

### STEP 1 — DISCOVER & LOAD ALL EXCEL FILES

PURPOSE:
  Automatically discover every .xlsx file in the raw_data folder (via
  pathlib, no hardcoded filenames), load each one, standardize its column
  names, and report per-file diagnostics before anything is merged.

WHY IT MATTERS FOR FATWA TEXTS:
  Fatwa datasets assembled from 21 separate exports are guaranteed to have
  schema drift (different header casing, different column ordering, the
  occasional extra column). Inspecting and standardizing each file BEFORE
  merging prevents a single malformed file from silently corrupting the
  merged corpus or from being dropped by a pd.concat column mismatch.

In [50]:
def discover_excel_files(folder: str) -> List[Path]:
    """Find every .xlsx file in `folder` using pathlib (no hardcoded names)."""
    root = Path(folder)
    if not root.exists():
        raise FileNotFoundError(f"Input folder not found: {folder}")

    files = sorted(root.glob("*.xlsx"))
    if not files:
        raise FileNotFoundError(f"No .xlsx files found in: {folder}")

    return files


def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercase + strip whitespace from every column name.

    Example: 'Question', 'QUESTION', ' question ' all become 'question'.
    This MUST run immediately after loading each file, before COLUMN_MAPPING
    is applied, since the mapping keys are defined in lowercase/stripped form.
    """
    df = df.rename(columns=lambda c: str(c).strip().lower())
    return df


def apply_column_mapping(df: pd.DataFrame, file_name: str) -> pd.DataFrame:
    """Rename standardized columns to title/question/answer via COLUMN_MAPPING,
    then keep ONLY the three canonical columns (missing ones are created as
    empty so every source file aligns to the same schema before concatenation).
    """
    rename_map = {c: COLUMN_MAPPING[c] for c in df.columns if c in COLUMN_MAPPING}
    unmapped = [c for c in df.columns if c not in COLUMN_MAPPING]
    if unmapped:
        logger.info(
            "  [%s] Ignoring unmapped column(s): %s", file_name, unmapped
        )

    df = df.rename(columns=rename_map)

    # Some files may legitimately be missing a column (e.g. no title column
    # at all) — create it empty rather than losing the file.
    for col in REQUIRED_FINAL_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    # If mapping produced duplicate target columns (e.g. two source columns
    # both mapped to "question"), keep the first non-null value per row.
    df = df.groupby(level=0, axis=1).first() if df.columns.duplicated().any() else df

    return df[REQUIRED_FINAL_COLUMNS]


def load_all_excel_files(folder: str) -> pd.DataFrame:
    """STEP 1: discover, load, standardize, and merge all 21 .xlsx files."""
    logger.info("=" * 70)
    logger.info("STEP 1: DISCOVERING AND LOADING EXCEL FILES")
    logger.info("=" * 70)

    files = discover_excel_files(folder)
    frames = []

    for path in files:
        raw = pd.read_excel(path)
        logger.info(
            "  File: %-30s | rows: %-6d | columns: %s",
            path.name, len(raw), list(raw.columns),
        )

        std = standardize_column_names(raw)
        mapped = apply_column_mapping(std, path.name)
        frames.append(mapped)

    merged = pd.concat(frames, ignore_index=True)

    logger.info("-" * 70)
    logger.info("STEP 1 SUMMARY:")
    logger.info("  Total files loaded   : %d", len(files))
    logger.info("  Total rows (merged)  : %d", len(merged))
    logger.info("  Final standardized columns: %s", list(merged.columns))

    return merged

### CONFIGURABLE COLUMN MAPPING

PURPOSE:
  The 21 source .xlsx files were produced by different scrapers/exports and
  do not share a single consistent header naming convention. This mapping
  is intentionally centralized and easy to extend: add a new key any time
  a 22nd file introduces yet another header spelling.

NOTE: keys are matched AFTER lowercasing + whitespace-stripping the raw
column name (see standardize_column_names below), so you only need to
list the lowercase form here.

In [51]:
COLUMN_MAPPING: Dict[str, str] = {
    # -> title
    "column1.title": "title",
    "column1.main_title": "title",
    "title": "title",
    "fatwa_title": "title",
    "main_title": "title",

    # -> question
    "column1.question": "question",
    "fatwa_question": "question",
    "question": "question",
    "q": "question",

    # -> answer
    "column1.answer": "answer",
    "fatwa_answer": "answer",
    "answer": "answer",
    "a": "answer",
}

REQUIRED_FINAL_COLUMNS = ["title", "question", "answer"]

### STEP 2 — CLEAN TEXT FIELDS (NO ROW DELETION FOR MISSING TITLE)

PURPOSE:
  Replace NaN with empty string and trim whitespace for title/question/
  answer. Rows are removed ONLY when question is empty OR answer is empty
  — a missing title is handled later by auto-generation (Step 3), never by
  deletion, since the question/answer still carry a valid, retrievable
  ruling even without a curated title.

WHY IT MATTERS:
  A fatwa with a missing question cannot be matched against a user query.
  A fatwa with a missing answer is retrievable but useless — worse, an
  empty answer can surface in top-k results and give a RAG generator
  nothing to ground on, causing hallucination on a religious ruling.

In [52]:
def clean_text_fields(df: pd.DataFrame) -> pd.DataFrame:
    for col in REQUIRED_FINAL_COLUMNS:
        df[col] = df[col].fillna("").astype(str).str.strip()
        # Excel sometimes serializes NaN as the literal string "nan"
        df[col] = df[col].apply(lambda v: "" if v.lower() == "nan" else v)
    return df


def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """STEP 2: drop rows only where question or answer is empty."""
    df = clean_text_fields(df)
    before = len(df)

    missing_question = (df["question"] == "").sum()
    missing_answer = (df["answer"] == "").sum()

    df = df[(df["question"] != "") & (df["answer"] != "")]
    after = len(df)

    logger.info("=" * 70)
    logger.info("STEP 2: MISSING VALUE HANDLING")
    logger.info("=" * 70)
    logger.info("  Rows removed (empty question): %d", missing_question)
    logger.info("  Rows removed (empty answer)  : %d", missing_answer)
    logger.info("  Total rows removed           : %d (%d -> %d)",
                 before - after, before, after)

    return df.reset_index(drop=True)

### STEP 3 — AUTO-GENERATE MISSING TITLES

PURPOSE:
  When title is empty, synthesize one from the first 10–15 words of the
  question, and flag the row via `generated_title` so downstream
  consumers (and human reviewers) can tell curated titles apart from
  synthetic ones.

WHY IT MATTERS FOR RETRIEVAL:
  BM25/RAG benefit from a short, high-signal "title" field that summarizes
  the fatwa topic. Leaving title empty wastes that retrieval signal for
  any record whose original scrape didn't include a curated title —
  generating one from the question recovers most of that signal cheaply.

In [53]:
MIN_TITLE_WORDS = 10
MAX_TITLE_WORDS = 15


def generate_title_from_question(question: str) -> str:
    words = question.split()
    n = min(MAX_TITLE_WORDS, len(words))
    n = max(n, min(MIN_TITLE_WORDS, len(words)))  # never exceed available words
    return " ".join(words[:n]).strip()


def apply_title_generation(df: pd.DataFrame) -> pd.DataFrame:
    df["generated_title"] = df["title"] == ""
    df.loc[df["generated_title"], "title"] = df.loc[
        df["generated_title"], "question"
    ].apply(generate_title_from_question)

    generated_count = df["generated_title"].sum()
    logger.info("=" * 70)
    logger.info("STEP 3: TITLE GENERATION")
    logger.info("=" * 70)
    logger.info("  Titles auto-generated from question: %d / %d rows",
                 generated_count, len(df))

    return df

### STEP 4 — REMOVE DUPLICATES (TITLE, THEN QUESTION, THEN ANSWER)

PURPOSE:
  Drop duplicate records in three sequential passes: exact-duplicate
  title, then exact-duplicate question, then exact-duplicate answer.

WHY IT MATTERS FOR BM25:
  BM25 scores documents using corpus-wide term-frequency statistics (IDF).
  If the same fatwa (or the same title/question) appears dozens of times
  — common when the same ruling is reposted across multiple Islamic
  sites and re-scraped into different source files — its terms get an
  artificially inflated document frequency, distorting IDF for the whole
  corpus and biasing retrieval toward duplicated topics. It also means a
  RAG system may surface 5 identical results in the top-10 instead of 5
  diverse, relevant fatwas.

In [54]:
def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    logger.info("=" * 70)
    logger.info("STEP 4: DUPLICATE REMOVAL")
    logger.info("=" * 70)

    before = len(df)

    # Use normalized fields instead of raw fields
    df = df.drop_duplicates(
        subset=[
            "question_norm",
            "answer_norm"
        ],
        keep="first"
    )

    removed = before - len(df)

    logger.info(
        "  Duplicate fatwas removed : %d",
        removed
    )

    logger.info(
        "  Final row count          : %d",
        len(df)
    )

    return df.reset_index(drop=True)

### STEP 5 — ARABIC TEXT NORMALIZATION

PURPOSE:
  Collapse Arabic letter variants that users almost never type consistently:
    أ / إ / آ  -> ا
    ى         -> ي
    ؤ         -> و   (NEW: hamza-on-waw)
    ئ         -> ي   (NEW: hamza-on-yeh)
  Strip tatweel (ـــ) and collapse extra whitespace.

  ة is intentionally NEVER normalized to ه. Religious terminology (e.g.
  الزكاة vs الزكاه) must be preserved exactly — merging teh marbuta with
  heh changes grammatical gender/word-class in a way that alef-variant
  merging does not, and it is common enough in fiqh vocabulary that the
  risk of meaning-loss outweighs any recall benefit.

WHY IT MATTERS FOR RETRIEVAL:
  BM25 is a purely lexical/term-matching algorithm — it has no semantic
  understanding. A user typing "اسلام" will never match "إسلام" unless
  both collapse to the same surface form first.

EXAMPLE:
  Before: "مـــا حكـم الإسلام في الصلاة على الآخرين ومؤمنيهم؟"
  After : "ما حكم الاسلام في الصلاة على الاخرين ومومنيهم؟"

In [55]:
def normalize_arabic(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Normalize alef variants
    text = re.sub(r"[إأآا]", "ا", text)
    # Normalize alef maksura to ya
    text = re.sub(r"ى", "ي", text)
    # Normalize hamza-on-waw / hamza-on-yeh
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    # Teh marbuta (ة) is intentionally NOT merged with heh (ه) — see docstring.

    # Remove tatweel (kashida elongation character)
    text = re.sub(r"ـ+", "", text)

    # NOTE: we deliberately do NOT run unicodedata NFKC normalization here.
    # NFKC's *compatibility* decomposition would expand devotional ligatures
    # such as ﷺ into their full spelled-out phrase, which violates the
    # requirement to preserve ﷺ / ﴿ / ﴾ / ؑ exactly as-is.

    # Collapse multiple whitespace/newlines into a single space
    text = re.sub(r"\s+", " ", text).strip()

    return text


def apply_normalization(df: pd.DataFrame) -> pd.DataFrame:
    sample_before = df["question"].iloc[0] if len(df) else ""

    df["title_norm"] = df["title"].apply(normalize_arabic)
    df["question_norm"] = df["question"].apply(normalize_arabic)
    df["answer_norm"] = df["answer"].apply(normalize_arabic)

    sample_after = df["question_norm"].iloc[0] if len(df) else ""
    logger.info("=" * 70)
    logger.info("STEP 5: ARABIC NORMALIZATION")
    logger.info("=" * 70)
    logger.info("  Before: %s", sample_before[:120])
    logger.info("  After : %s", sample_after[:120])

    return df

### STEP 6 — REMOVE ARABIC DIACRITICS (TASHKEEL)

PURPOSE:
  Strip diacritical marks: Fatha, Damma, Kasra, Shadda, Sukun, Tanween.

WHY IT MATTERS:
  Diacritics carry pronunciation/grammatical detail, but almost no users
  type them when searching, and scraped fatwa answers are inconsistent
  about including them. If left in the corpus, an identical word with and
  without diacritics is treated as two different tokens by BM25 — silently
  splitting term frequency and hurting recall.

EXAMPLE:
  Before: "الصَّلَاةُ فِي وَقْتِهَا وَاجِبَةٌ"
  After : "الصلاة في وقتها واجبة"

In [56]:
ARABIC_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")


def remove_diacritics(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return ARABIC_DIACRITICS.sub("", text)


def apply_diacritics_removal(df: pd.DataFrame) -> pd.DataFrame:
    sample_before = df["answer_norm"].iloc[0] if len(df) else ""

    df["title_norm"] = df["title_norm"].apply(remove_diacritics)
    df["question_norm"] = df["question_norm"].apply(remove_diacritics)
    df["answer_norm"] = df["answer_norm"].apply(remove_diacritics)

    sample_after = df["answer_norm"].iloc[0] if len(df) else ""
    logger.info("=" * 70)
    logger.info("STEP 6: DIACRITICS REMOVAL")
    logger.info("=" * 70)
    logger.info("  Before: %s", sample_before[:120])
    logger.info("  After : %s", sample_after[:120])

    return df

### STEP 7 — REMOVE ONLY NOISY NON-ARABIC SYMBOLS (KEEP SENTENCE PUNCTUATION)

PURPOSE:
  Strip scraping artifacts (bullets, HTML leftovers, stray symbols)
  WITHOUT deleting meaningful Arabic words, digits, sentence-boundary
  punctuation, or devotional markers.

WHAT IS DELIBERATELY KEPT (unlike the earlier single-file version, which
stripped ALL punctuation):
    .  ،  ؛  :  ؟   — sentence/clause boundary markers needed for future
                       sentence segmentation, chunking, passage retrieval,
                       and citation extraction.
    ﴿ ﴾              — Qur'anic quotation delimiters
    ﷺ                — devotional ligature (peace be upon him)
    ؑ                — devotional diacritic-like mark ('alayhi al-salam)

WHY IT MATTERS:
  Scraped fatwa pages routinely leave in artifacts like "●", "»فتوى«", or
  repeated navigation punctuation. These add noise to the BM25 vocabulary.
  However, blindly stripping ALL punctuation (as a naive cleaner would)
  destroys the sentence boundaries a downstream chunker needs to split a
  long answer into coherent passages, and can strip Qur'an delimiters that
  help a reviewer spot embedded scripture for verification.

In [57]:
ARABIC_LETTERS = r"\u0621-\u064A"  # basic Arabic letters (post alef-normalization)
PRESERVED_PUNCTUATION = ".,،؛:؟"
PRESERVED_SYMBOLS = "﴿﴾ﷺؑ"
_KEEP_CHARS = re.escape(PRESERVED_PUNCTUATION + PRESERVED_SYMBOLS)
KEEP_PATTERN = re.compile(rf"[^{ARABIC_LETTERS}0-9\s{_KEEP_CHARS}]")


def remove_special_characters(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = KEEP_PATTERN.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def apply_special_char_removal(df: pd.DataFrame) -> pd.DataFrame:
    sample_before = df["question_norm"].iloc[0] if len(df) else ""

    df["title_norm"] = df["title_norm"].apply(remove_special_characters)
    df["question_norm"] = df["question_norm"].apply(remove_special_characters)
    df["answer_norm"] = df["answer_norm"].apply(remove_special_characters)

    sample_after = df["question_norm"].iloc[0] if len(df) else ""
    logger.info("=" * 70)
    logger.info("STEP 7: NOISY-SYMBOL REMOVAL (punctuation & devotional marks kept)")
    logger.info("=" * 70)
    logger.info("  Before: %s", sample_before[:120])
    logger.info("  After : %s", sample_after[:120])

    return df

### STEP 8 — RELIGIOUS-TEXT SAFETY CHECK (safe_no_negation_loss)

PURPOSE:
  This step is deliberately a NO-OP filter/validator, not an active
  transformer. Its job is to flag any record where cleaning appears to
  have dropped a negation particle from the answer — kept EXACTLY as
  implemented in the original pipeline; logic is not modified.

RISKS OF OVER-CLEANING ISLAMIC TEXT:
  - "لا يجوز" (not permissible) vs "يجوز" (permissible) — losing "لا"
    flips the ruling entirely. We never remove لا / لم / لن / ما / ليس /
    غير even in the stopword-removed tokenized version (Step 10).
  - The ORIGINAL, untouched title/question/answer columns are always kept
    alongside the normalized ones in the export, so a scholar/reviewer can
    always verify the cleaned version didn't distort the ruling.

In [58]:
NEGATION_PARTICLES = {"لا", "لم", "لن", "ما", "ليس", "غير"}


def validate_no_negation_loss(original: str, cleaned: str) -> bool:
    """Returns True if the record is SAFE (no negation particle was lost)."""
    orig_tokens = set(str(original).split())
    clean_tokens = set(str(cleaned).split())
    lost_negations = NEGATION_PARTICLES & orig_tokens - clean_tokens
    return len(lost_negations) == 0


def apply_religious_text_safety_check(df: pd.DataFrame) -> pd.DataFrame:
    df["safe_no_negation_loss"] = df.apply(
        lambda r: validate_no_negation_loss(r["answer"], r["answer_norm"]), axis=1
    )
    flagged = (~df["safe_no_negation_loss"]).sum()
    logger.info("=" * 70)
    logger.info("STEP 8: RELIGIOUS-TEXT SAFETY CHECK")
    logger.info("=" * 70)
    logger.info(
        "  %d record(s) flagged for manual review (possible negation loss).",
        flagged,
    )
    if flagged > 0:
        logger.info("  -> Inspect these rows before trusting the cleaned answer_norm field.")
    return df

### STEP 9 — TOKENIZATION

PURPOSE:
  Split normalized text into tokens via whitespace tokenization (safe now
  that diacritics are stripped, and punctuation is retained as separate
  standalone tokens rather than fused into words).

WHY BM25 REQUIRES TOKENIZED DOCUMENTS:
  BM25 is a bag-of-words ranking function built entirely on term
  frequency (TF) / inverse document frequency (IDF) computed over
  discrete tokens. Libraries like `rank_bm25` expect `List[List[str]]`.

In [59]:
def tokenize(text: str) -> list:
    if not isinstance(text, str):
        return []
    return text.split()


def apply_tokenization(df: pd.DataFrame) -> pd.DataFrame:
    df["title_tokens"] = df["title_norm"].apply(tokenize)
    df["question_tokens"] = df["question_norm"].apply(tokenize)
    df["answer_tokens"] = df["answer_norm"].apply(tokenize)

    logger.info("=" * 70)
    logger.info("STEP 9: TOKENIZATION")
    logger.info("=" * 70)
    if len(df):
        logger.info("  Example question tokens: %s", df["question_tokens"].iloc[0][:10])

    return df

### STEP 10 — ARABIC STOPWORD ANALYSIS (TWO VERSIONS)

PURPOSE:
  Produce Version A (no stopword removal) and Version B (conservative
  stopword removal) so both can be benchmarked for BM25 retrieval quality.

CRITICAL — NEVER REMOVE THESE, EVEN IN VERSION B:
    على، في، عن   — prepositions that frequently carry the legal/topical
                     distinction in a fatwa question (e.g. "الصلاة على
                     الميت" vs "الصلاة في المسجد" — the preposition
                     changes what is even being asked about).
    لا، لم، لن، ما — negation particles. Removing any of these can flip a
                     ruling from prohibited to permitted or vice versa
                     (see Step 8's safety check, which guards this too).
  These six words are excluded from ARABIC_STOPWORDS below by design —
  do not "clean up" this list by adding them back in.

In [60]:
ARABIC_STOPWORDS = {
    "من", "الى", "إلى", "مع", "هذا", "هذه", "ذلك",
    "التي", "الذي", "الذين", "و", "او", "أو", "ثم", "قد", "كل", "كان",
    "كانت", "يكون", "هو", "هي", "هم", "انت", "أنت", "انا", "أنا", "نحن",
}

# Defensive assertion: guarantee the protected words can never accidentally
# end up in the stopword set (e.g. via a future careless edit / merge).
_PROTECTED_WORDS = {"على", "في", "عن", "لا", "لم", "لن", "ما"}
assert not (ARABIC_STOPWORDS & _PROTECTED_WORDS), (
    "A protected preposition/negation particle was found in ARABIC_STOPWORDS. "
    "These words must never be removed — they can change a fatwa's meaning."
)


def remove_stopwords(tokens: list) -> list:
    return [t for t in tokens if t not in ARABIC_STOPWORDS]


def apply_stopword_versions(df: pd.DataFrame) -> pd.DataFrame:
    # Version A: no stopword removal (BM25-ready as-is)
    df["question_tokens_A"] = df["question_tokens"]
    df["answer_tokens_A"] = df["answer_tokens"]

    # Version B: conservative stopword removal (protected words excluded)
    df["question_tokens_B"] = df["question_tokens"].apply(remove_stopwords)
    df["answer_tokens_B"] = df["answer_tokens"].apply(remove_stopwords)

    logger.info("=" * 70)
    logger.info("STEP 10: STOPWORD VERSIONS A (kept) AND B (removed) CREATED")
    logger.info("=" * 70)
    logger.info(
        "  Version A = no stopword removal (recommended default for BM25 on "
        "short fatwa questions, since IDF already discounts frequent terms)."
    )
    logger.info(
        "  Version B = stopword removal EXCLUDING على/في/عن/لا/لم/لن/ما "
        "(smaller index; validate with NDCG@10 before using in production)."
    )
    return df

### STEP 11 — BUILD STRUCTURED RETRIEVAL DOCUMENT

PURPOSE:
  Assemble retrieval_title / retrieval_question / retrieval_answer (the
  normalized title/question/answer) into one structured retrieval_document
  with explicit [Title] / [Question] / [Answer] sections.

WHY THIS IMPROVES BM25 RETRIEVAL:
  Titles often summarize the fatwa topic in a few dense, high-IDF terms
  that may not otherwise repeat verbatim in the question or answer body.
  Including the title as its own labeled section lets BM25 credit a query
  match against the topic summary, not just the full-text body — this
  materially improves ranking for short, topic-style user queries.

WHY THIS IMPROVES RAG:
  The [Title]/[Question]/[Answer] structure gives the generation model a
  clean, unambiguous grounding context: it can distinguish "what was
  asked" from "what was ruled" instead of receiving one undifferentiated
  blob of text, reducing the chance the model conflates the question's
  premise with the actual fatwa ruling.

In [61]:
def build_retrieval_document(row) -> str:
    return (
        f"[Title]\n{row['retrieval_title']}\n\n"
        f"[Question]\n{row['retrieval_question']}\n\n"
        f"[Answer]\n{row['retrieval_answer']}"
    ).strip()


def apply_build_retrieval_text(df: pd.DataFrame) -> pd.DataFrame:
    df["retrieval_title"] = df["title_norm"]
    df["retrieval_question"] = df["question_norm"]
    df["retrieval_answer"] = df["answer_norm"]
    df["retrieval_document"] = df.apply(build_retrieval_document, axis=1)

    logger.info("=" * 70)
    logger.info("STEP 11: STRUCTURED RETRIEVAL DOCUMENT BUILT")
    logger.info("=" * 70)
    if len(df):
        logger.info("  Example:\n%s", df["retrieval_document"].iloc[0][:200])

    return df

### STEP 12 — CHUNKING FLAG (needs_chunking)

PURPOSE:
  Flag any fatwa whose answer exceeds 300 words as needing chunking before
  embedding for the RAG vector store — long answers embedded as a single
  vector dilute the semantic signal and hurt retrieval precision for
  queries about a specific sub-point buried inside a long ruling.

In [62]:
CHUNKING_WORD_THRESHOLD = 300


def apply_chunking_flag(df: pd.DataFrame) -> pd.DataFrame:
    df["needs_chunking"] = df["answer"].apply(lambda t: len(str(t).split()) > CHUNKING_WORD_THRESHOLD)

    long_count = df["needs_chunking"].sum()
    pct = (long_count / len(df) * 100) if len(df) else 0.0

    logger.info("=" * 70)
    logger.info("STEP 12: CHUNKING FLAG")
    logger.info("=" * 70)
    logger.info("  Long fatwas (answer > %d words): %d", CHUNKING_WORD_THRESHOLD, long_count)
    logger.info("  Percentage of long fatwas       : %.2f%%", pct)

    return df

### STEP 13 — PRE-SAVE CORPUS STATISTICS

In [63]:
def print_corpus_statistics(df: pd.DataFrame) -> None:
    def avg_words(series: pd.Series) -> float:
        return series.apply(lambda t: len(str(t).split())).mean() if len(series) else 0.0

    answer_word_counts = df["answer"].apply(lambda t: len(str(t).split()))

    logger.info("=" * 70)
    logger.info("STEP 13: PRE-SAVE CORPUS STATISTICS")
    logger.info("=" * 70)
    logger.info("  Total documents        : %d", len(df))
    logger.info("  Avg title length (words)   : %.2f", avg_words(df["title"]))
    logger.info("  Avg question length (words): %.2f", avg_words(df["question"]))
    logger.info("  Avg answer length (words)  : %.2f", avg_words(df["answer"]))
    logger.info("  Min answer length (words)  : %d",
                 int(answer_word_counts.min()) if len(df) else 0)
    logger.info("  Max answer length (words)  : %d",
                 int(answer_word_counts.max()) if len(df) else 0)

### STEP 14 — SAVE CLEAN DATASET (.xlsx AND .csv)

PURPOSE:
  Export the cleaned dataframe to BOTH clean_fatwas.xlsx and
  clean_fatwas.csv (utf-8-sig), with a fixed, minimal, audit-friendly
  column set — no intermediate token/debug columns are exported.

WHY PRESERVING THE CLEANED DATASET MATTERS:
  - Reproducibility: every downstream component (BM25 index, embeddings
    for the RAG vector store, the QA UI) should read from ONE frozen,
    versioned cleaned file rather than re-running cleaning with slightly
    different logic each time — this prevents silent drift between what
    was indexed and what gets displayed to users.
  - Auditability: since these are religious rulings, any answer shown to
    a user must be traceable back to the exact original title/question/
    answer. We keep the RAW original columns alongside the normalized
    ones (never overwriting title/question/answer), so a scholar or QA
    reviewer can always verify the cleaned version didn't distort a ruling.

In [64]:
EXPORT_COLUMNS = [
    "title", "question", "answer",                # original (untouched)
    "title_norm", "question_norm", "answer_norm",  # normalized text
    "retrieval_document",                          # BM25/RAG input
    "generated_title",                             # provenance flag
    "safe_no_negation_loss",                        # QA safety flag
    "needs_chunking",                               # RAG chunking flag
]


def save_clean_dataset(df: pd.DataFrame, output_dir: str) -> None:
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    export_columns = [c for c in EXPORT_COLUMNS if c in df.columns]
    export_df = df[export_columns]

    xlsx_path = out_dir / "clean_fatwas.xlsx"
    csv_path = out_dir / "clean_fatwas.csv"

    export_df.to_excel(xlsx_path, index=False)
    export_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    logger.info("=" * 70)
    logger.info("STEP 14: CLEAN DATASET SAVED")
    logger.info("=" * 70)
    logger.info("  %s (%d rows, %d columns)", xlsx_path, len(export_df), len(export_columns))
    logger.info("  %s (%d rows, %d columns)", csv_path, len(export_df), len(export_columns))

### PIPELINE ORCHESTRATION

In [65]:
def run_pipeline(input_folder: str, output_dir: str) -> pd.DataFrame:
    df = load_all_excel_files(input_folder)
    df = handle_missing_values(df)
    df = apply_title_generation(df)
    df = apply_normalization(df)
    df = apply_diacritics_removal(df)
    df = apply_special_char_removal(df)
    df = remove_duplicates(df)
    df = apply_religious_text_safety_check(df)
    df = apply_tokenization(df)
    df = apply_stopword_versions(df)
    df = apply_build_retrieval_text(df)
    df = apply_chunking_flag(df)
    print_corpus_statistics(df)
    save_clean_dataset(df, output_dir)
    return df

## Configuration & Execution

Set `INPUT_FOLDER` to the folder containing your 21 raw `.xlsx` fatwa files, and `OUTPUT_DIR` to where `clean_fatwas.xlsx` / `clean_fatwas.csv` should be written. Then run the cell below to execute the full pipeline.

In [66]:
INPUT_FOLDER = "/content"
OUTPUT_DIR = "/content/output"

clean_df = run_pipeline(INPUT_FOLDER, OUTPUT_DIR)

INFO:fatwa_pipeline:======================================================================
INFO:fatwa_pipeline:STEP 1: DISCOVERING AND LOADING EXCEL FILES
INFO:fatwa_pipeline:======================================================================
INFO:fatwa_pipeline:  File: Al Shaikh Abdual Aziz Al Ashaikh_after removing frequent words.xlsx | rows: 135    | columns: ['Column1.title', 'Column1.question', 'Column1.answer']
INFO:fatwa_pipeline:  File: Al Shaikh Abdual Aziz Ibn Baz_after removing frequent words.xlsx | rows: 27111  | columns: ['Column1.main_title_fiqhi', 'Column1.main_title_moudu3e', 'Column1.sub_title', 'Column1.question', 'Column1.answer', 'Column1.fatwa_owner', 'Column1._class', 'Column1.website_name']
INFO:fatwa_pipeline:  [Al Shaikh Abdual Aziz Ibn Baz_after removing frequent words.xlsx] Ignoring unmapped column(s): ['column1.main_title_fiqhi', 'column1.main_title_moudu3e', 'column1.sub_title', 'column1.fatwa_owner', 'column1._class', 'column1.website_name']
INFO:fatwa_

In [28]:
print(clean_df.shape)

import os
print(os.path.exists("/content/output/clean_fatwas.xlsx"))
print(os.path.exists("/content/output/clean_fatwas.csv"))

!ls /content/output

(117809, 20)
True
True
clean_fatwas.csv  clean_fatwas.xlsx


In [24]:
!find /content -name "*.xlsx"

/content/fatwapedia part 1.xlsx
/content/Al Shaikh Mohammad Ibn Othaimin_after removing frequent words.xlsx
/content/ahkam which is researches that has not been used.xlsx
/content/islamQA_after removing frequent words.xlsx
/content/Dar Al Ifta in Saudi Arabia_ Fhrs section.xlsx
/content/Dar Al Ifta in Saudi Arabia_Al lajna section.xlsx
/content/Al Shaikh Abdual Aziz Al Ashaikh_after removing frequent words.xlsx
/content/Dar Al Ifta in Saudi Arabia_Nour Ala Addarb section_after removing frequent words.xlsx
/content/Dar Al Ifta in Egypt_after removing frequent words.xlsx
/content/‏‏‏‏fatwapedia part 2.xlsx
/content/islamweb_after removing frequent words.xlsx
/content/Al Shaikh Abdual Aziz Ibn Baz_after removing frequent words.xlsx
/content/Al Shaikh Saleh Al Fwzan_after removing frequent words.xlsx
/content/Al Shaikh Saleh Bin Humaid.xlsx
/content/Dar Al Ifta in Jordan_after removing frequent words.xlsx
/content/Al Shaikh Abdullah Al Manee_after removing frequent words.xlsx
/content/Dar 

In [47]:
import logging

logging.basicConfig(
    level=logging.INFO,
    force=True
)

logger = logging.getLogger(__name__)

logger.info("TEST LOGGER")
print("TEST PRINT")

INFO:__main__:TEST LOGGER


TEST PRINT


In [48]:
print(logger)

<Logger __main__ (INFO)>
